# ROBUST04 Run 2: Hybrid Retrieval + Reciprocal Rank Fusion

## Strategy
- Combine multiple retrieval methods:
  1. BM25 (optimized)
  2. BM25 + RM3 (different parameters)
  3. Query Language Model (Dirichlet)
- Apply Reciprocal Rank Fusion (RRF) to merge rankings
- Expected MAP: 0.27-0.29 (2-5% improvement over single method)

In [ ]:
# ============================================================================
# JAVA 21 SETUP - Add this BEFORE pip install pyserini
# ============================================================================

import os
import subprocess

print("Checking Java version...")

# Check current Java version
try:
    result = subprocess.run(['java', '-version'], capture_output=True, text=True)
    current_version = result.stderr
    print(f"Current Java:\n{current_version}")
except:
    print("Java not found")

# Install Java 21
print("\n📥 Installing Java 21...")
!sudo apt-get update -qq
!sudo apt-get install -y openjdk-21-jdk-headless -qq

# Set Java 21 as default
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]

# Verify installation
print("\n✓ Verifying Java 21 installation...")
!java -version

print("\n" + "="*70)
print("Java 21 installed! Now you can install pyserini.")
print("="*70)
# ============================================================================

In [ ]:
# ============================================================================
# INSTALL COMPATIBLE VERSIONS
# ============================================================================

print("Installing compatible package versions...")

# Uninstall conflicting versions first
!pip uninstall -y pyserini transformers -q

# Install specific compatible versions
!pip install -q transformers==4.41.2
!pip install -q pyserini==0.22.1

# Install other dependencies
!pip install -q ir_measures torch torchvision torchaudio sentence-transformers accelerate google-generativeai #numpy

#install Faiss
!pip install faiss-cpu --no-cache

print("✓ All packages installed with compatible versions!")
print("\nVerifying installations...")

In [ ]:
import os
import transformers
import pyserini
from collections import defaultdict
from pyserini.search.lucene import LuceneSearcher
from pyserini.index.lucene import IndexReader
import ir_measures
from ir_measures import MAP, nDCG, P
import math

In [ ]:
# ============================================================================
# GOOGLE DRIVE SETUP - Add this as the first cell
# ============================================================================
from google.colab import drive, userdata
import os
import sys

# Check if running in Google Colab
try:
    IN_COLAB = True
    print("✓ Running in Google Colab")
except:
    IN_COLAB = False
    print("✓ Running locally")

# Mount Google Drive if in Colab
if IN_COLAB:
    print("\nMounting Google Drive...")
    drive.mount('/content/drive')
    print("✓ Google Drive mounted")

    # Set base directory (CHANGE THIS to match your folder!)
    BASE_DIR = '/content/drive/MyDrive/TEXT/FinalProject'

    # Check if directory exists
    if os.path.exists(BASE_DIR):
        print(f"✓ Found directory: {BASE_DIR}")
        os.chdir(BASE_DIR)
        print(f"✓ Changed to: {os.getcwd()}")
    else:
        print(f"⚠ Directory not found: {BASE_DIR}")
        print(f"  Please update BASE_DIR to match your folder")
else:
    # Running locally
    BASE_DIR = os.getcwd()
    print(f"Working directory: {BASE_DIR}")

# Verify files exist
print("\n📁 Checking for required files...")
if os.path.exists('queriesROBUST.txt'):
    print(f"  ✓ Found: queriesROBUST.txt")
else:
    print(f"  ✗ Missing: queriesROBUST.txt")

if os.path.exists('qrels_50_Queries'):
    print(f"  ✓ Found: qrels_50_Queries")
else:
    print(f"  ✗ Missing: qrels_50_Queries")

print("\n" + "="*70)
print("Setup complete! Continue with the notebook below.")
print("="*70)


## 1. Load ROBUST04 Index

In [ ]:
print("Loading ROBUST04 index...")
index_reader = IndexReader.from_prebuilt_index('robust04')

# Create three separate searchers for different methods
searcher_bm25 = LuceneSearcher.from_prebuilt_index('robust04')
searcher_rm3 = LuceneSearcher.from_prebuilt_index('robust04')
searcher_ql = LuceneSearcher.from_prebuilt_index('robust04')

print(f"Index contains {index_reader.stats()['documents']} documents")

## 2. Load Queries and QRELs

In [ ]:
def load_queries(query_file):
    queries = {}
    with open(query_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(None, 1)
            if len(parts) == 2:
                qid, query_text = parts
                queries[qid] = query_text
    return queries

def load_qrels(qrel_file):
    qrels = defaultdict(dict)
    with open(qrel_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qid = parts[0]
                docid = parts[2]
                rel = int(parts[3])
                qrels[qid][docid] = rel
    return qrels

# Load data
queries = load_queries('queriesROBUST.txt')
qrels = load_qrels('qrels_50_Queries')

train_qids = sorted(qrels.keys())
test_qids = sorted([qid for qid in queries.keys() if qid not in train_qids])

print(f"Loaded {len(queries)} queries")
print(f"Training queries: {len(train_qids)}")
print(f"Test queries: {len(test_qids)}")

## 3. Reciprocal Rank Fusion (RRF) Implementation

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Combine multiple ranked lists using Reciprocal Rank Fusion.
    
    Args:
        ranked_lists: List of ranked lists, where each list contains tuples (docid, score, rank)
        k: RRF constant (typically 60)
    
    Returns:
        List of (docid, rrf_score) tuples sorted by RRF score (descending)
    """
    rrf_scores = defaultdict(float)
    
    for ranked_list in ranked_lists:
        for docid, original_score, rank in ranked_list:
            rrf_scores[docid] += 1.0 / (k + rank)
    
    # Sort by RRF score (descending)
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_docs

def weighted_rrf(ranked_lists, weights, k=60):
    """
    Weighted version of RRF for giving different importance to methods.
    
    Args:
        ranked_lists: List of ranked lists
        weights: List of weights (same length as ranked_lists)
        k: RRF constant
    """
    rrf_scores = defaultdict(float)
    
    for ranked_list, weight in zip(ranked_lists, weights):
        for docid, original_score, rank in ranked_list:
            rrf_scores[docid] += weight * (1.0 / (k + rank))
    
    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_docs

## 4. Multi-Method Retrieval Function

In [ ]:
def retrieve_multiple_methods(query_text, k=1000):
    """
    Retrieve using three different methods.
    
    Returns:
        List of three ranked lists (one per method)
    """
    ranked_lists = []
    
    # Method 1: BM25 (optimized parameters)
    hits_bm25 = searcher_bm25.search(query_text, k=k)
    list_bm25 = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_bm25, start=1)]
    ranked_lists.append(list_bm25)
    
    # Method 2: BM25 + RM3
    hits_rm3 = searcher_rm3.search(query_text, k=k)
    list_rm3 = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_rm3, start=1)]
    ranked_lists.append(list_rm3)
    
    # Method 3: Query Language Model (Dirichlet)
    hits_ql = searcher_ql.search(query_text, k=k)
    list_ql = [(hit.docid, hit.score, rank) for rank, hit in enumerate(hits_ql, start=1)]
    ranked_lists.append(list_ql)
    
    return ranked_lists

## 5. Evaluation Function for RRF

In [ ]:
def evaluate_rrf(query_dict, qrels_dict, rrf_k=60, use_weights=False, weights=None, top_k=1000):
    """
    Evaluate RRF on given queries.
    """
    run_results = []
    
    for qid in query_dict:
        if qid not in qrels_dict:
            continue
        
        query_text = query_dict[qid]
        
        # Retrieve from multiple methods
        ranked_lists = retrieve_multiple_methods(query_text, k=top_k)
        
        # Apply RRF
        if use_weights and weights is not None:
            fused_results = weighted_rrf(ranked_lists, weights, k=rrf_k)
        else:
            fused_results = reciprocal_rank_fusion(ranked_lists, k=rrf_k)
        
        # Take top 1000
        for rank, (docid, rrf_score) in enumerate(fused_results[:top_k], start=1):
            run_results.append(ir_measures.ScoredDoc(
                query_id=qid,
                doc_id=docid,
                score=rrf_score
            ))
    
    # Convert qrels
    qrels_list = []
    for qid, doc_rels in qrels_dict.items():
        for docid, rel in doc_rels.items():
            qrels_list.append(ir_measures.Qrel(
                query_id=qid,
                doc_id=docid,
                relevance=rel
            ))
    
    # Evaluate
    metrics = [MAP, nDCG@10, P@10]
    results = ir_measures.calc_aggregate(metrics, qrels_list, run_results)
    
    return results

## 6. Configure Individual Searchers

### Method 1: BM25 with tuned parameters

In [ ]:
# Configure BM25 searcher with optimized parameters
# (Use parameters from Run 1, or set to reasonable defaults)
searcher_bm25.set_bm25(k1=1.2, b=0.75)
print("Method 1: BM25 (k1=1.2, b=0.75)")

### Method 2: BM25 + RM3 with different parameters

In [ ]:
# Configure RM3 searcher with different parameters than Run 1
searcher_rm3.set_bm25(k1=0.9, b=0.4)
searcher_rm3.set_rm3(fb_docs=20, fb_terms=60, original_query_weight=0.5)
print("Method 2: BM25+RM3 (k1=0.9, b=0.4, fb_docs=20, fb_terms=60, orig_weight=0.5)")

### Method 3: Query Language Model (Dirichlet)

In [ ]:
# Configure QL searcher with Dirichlet smoothing
searcher_ql.set_qld(mu=1000)  # mu is the Dirichlet prior parameter
print("Method 3: Query Language Model - Dirichlet (mu=1000)")

## 7. Tune RRF Parameters on Training Set

In [ ]:
# Create train query dict
train_queries = {qid: queries[qid] for qid in train_qids}

# Test different RRF k values
rrf_k_values = [30, 60, 90]

print("Tuning RRF constant k...\n")
best_rrf_k = 60
best_map = 0

for rrf_k in rrf_k_values:
    print(f"Testing RRF k={rrf_k}...")
    results = evaluate_rrf(train_queries, qrels, rrf_k=rrf_k, use_weights=False)
    map_score = results[MAP]
    
    print(f"  MAP: {map_score:.4f}")
    print(f"  nDCG@10: {results[nDCG@10]:.4f}")
    print(f"  P@10: {results[P@10]:.4f}\n")
    
    if map_score > best_map:
        best_map = map_score
        best_rrf_k = rrf_k

print(f"Best RRF k: {best_rrf_k} with MAP: {best_map:.4f}")

## 8. Optional: Test Weighted RRF

In [ ]:
# Test different weight combinations
weight_combinations = [
    [1.0, 1.0, 1.0],  # Equal weights (standard RRF)
    [1.5, 1.0, 0.5],  # Favor BM25
    [1.0, 1.5, 0.5],  # Favor RM3
    [0.8, 1.2, 1.0],  # Balanced
]

print("Testing weighted RRF...\n")
best_weights = [1.0, 1.0, 1.0]
best_weighted_map = best_map

for weights in weight_combinations:
    print(f"Testing weights: {weights}")
    results = evaluate_rrf(train_queries, qrels, rrf_k=best_rrf_k, use_weights=True, weights=weights)
    map_score = results[MAP]
    
    print(f"  MAP: {map_score:.4f}\n")
    
    if map_score > best_weighted_map:
        best_weighted_map = map_score
        best_weights = weights

print(f"Best weights: {best_weights} with MAP: {best_weighted_map:.4f}")

# Decide whether to use weights
use_weighted = best_weighted_map > best_map
if use_weighted:
    print(f"\nUsing weighted RRF (improvement: {best_weighted_map - best_map:.4f})")
    final_weights = best_weights
    final_map = best_weighted_map
else:
    print(f"\nUsing standard RRF (no improvement from weighting)")
    final_weights = None
    final_map = best_map

## 9. Generate Run 2 for Test Queries (199 queries)

In [ ]:
print(f"Generating run_2.res using RRF...")
print(f"RRF k: {best_rrf_k}")
if use_weighted:
    print(f"Weights: {final_weights}")
print(f"Number of test queries: {len(test_qids)}")

run2_results = []

for i, qid in enumerate(test_qids, 1):
    query_text = queries[qid]
    
    # Retrieve from multiple methods
    ranked_lists = retrieve_multiple_methods(query_text, k=1000)
    
    # Apply RRF
    if use_weighted:
        fused_results = weighted_rrf(ranked_lists, final_weights, k=best_rrf_k)
    else:
        fused_results = reciprocal_rank_fusion(ranked_lists, k=best_rrf_k)
    
    # Store top 1000 results
    for rank, (docid, rrf_score) in enumerate(fused_results[:1000], start=1):
        run2_results.append({
            'qid': qid,
            'docid': docid,
            'rank': rank,
            'score': rrf_score
        })
    
    if i % 20 == 0:
        print(f"Processed {i}/{len(test_qids)} queries")

print(f"\nTotal results: {len(run2_results)}")

## 10. Save Run 2 in TREC Format

In [ ]:
def save_trec_run(results, output_file, run_tag):
    with open(output_file, 'w') as f:
        for result in results:
            f.write(f"{result['qid']} Q0 {result['docid']} {result['rank']} {result['score']:.6f} {run_tag}\n")
    print(f"Saved {len(results)} results to {output_file}")

# Save Run 2
save_trec_run(run2_results, 'run_2.res', 'run2')

# Verify file format
print("\nFirst 5 lines of run_2.res:")
with open('run_2.res', 'r') as f:
    for i, line in enumerate(f):
        if i < 5:
            print(line.strip())
        else:
            break

## 11. Summary Statistics

In [ ]:
print("\n" + "="*60)
print("RUN 2 SUMMARY - HYBRID RETRIEVAL + RRF")
print("="*60)
print(f"Best MAP on training set (50 queries): {final_map:.4f}")
print(f"\nRetrieval Methods:")
print(f"  1. BM25 (k1=1.2, b=0.75)")
print(f"  2. BM25+RM3 (k1=0.9, b=0.4, fb_docs=20, fb_terms=60)")
print(f"  3. Query Language Model (Dirichlet, mu=1000)")
print(f"\nFusion Method: {'Weighted RRF' if use_weighted else 'Standard RRF'}")
print(f"  RRF constant k: {best_rrf_k}")
if use_weighted:
    print(f"  Weights: {final_weights}")
print(f"\nTest queries processed: {len(test_qids)}")
print(f"Total documents ranked: {len(run2_results)}")
print(f"Average docs per query: {len(run2_results)/len(test_qids):.1f}")
print(f"\nOutput file: run_2.res")
print("="*60)